In [2]:
pip install pandasql

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=4c6011f5c08cbe77591381ffa17e2924c319aae26cf8e16972791d05c5b9d431
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [3]:
import pandas as pd
from pandasql import sqldf

In [4]:
pysqldf = lambda q: sqldf(q, globals())


In [5]:
buyers = pd.read_csv('/content/buyers.csv', sep=',')
order_items = pd.read_csv('/content/order_items.csv', sep=',')
orders = pd.read_csv('/content/orders.csv', sep=',')
payments = pd.read_csv('/content/payments.csv', sep=',')
products = pd.read_csv('/content/products.csv', sep=',')
sellers = pd.read_csv('/content/sellers.csv', sep=',')

In [6]:
display(buyers.head())
display(order_items.head())
display(orders.head())
display(payments.head())
display(products.head())
display(sellers.head())


,id;name;city;state;segment;created_at
0,1;Nascimento & Costa Distribuidora Mercearia;R...
1,2;Lima & Almeida Atacado Mercearia;Rio de Jane...
2,3;Ferreira & Costa Grupo Mercearia;São Paulo;P...
3,4;Lima & Almeida Comércio Supermercado;Campo G...
4,5;Ferreira & Ferreira Comércio Mercearia;São P...


,id,order_id,product_id,qty,unit_price,discount
0,1,1,500,6,240.06,117.48
1,2,1,778,6,126.47,94.54
2,3,2,346,39,492.08,98.86
3,4,2,248,20,278.55,588.33
4,5,3,701,9,298.75,272.33


,id,seller_id,buyer_id,status,created_at,total_value
0,1,114,2806,delivered,2023-08-22 05:25:59,1987.16
1,2,90,2586,processing,2024-07-19 03:57:54,24074.93
2,3,96,1849,completed,2024-06-14 22:16:15,2416.42
3,4,10,116,processing,2024-07-11 04:46:30,3871.48
4,5,16,2195,delivered,2023-09-07 20:06:40,15367.65


,id,order_id,paid_at,amount,method,status
0,1,1,2023-08-23 19:25:59,1987.16,transfer,paid
1,2,2,2024-07-20 17:57:54,24074.93,boleto,paid
2,3,3,2024-06-16 10:16:15,2416.42,transfer,paid
3,4,4,2024-07-11 11:46:30,3871.48,boleto,paid
4,5,5,2023-09-08 00:06:40,15367.65,transfer,paid


,id,name,category,seller_id,active,unit_cost
0,1,Produto Laticínios Linha 1,Snacks,31,1,104.39
1,2,Produto Grãos Linha 2,Carnes,71,1,153.19
2,3,Produto Enlatados Linha 3,Grãos,9,0,90.57
3,4,Produto Snacks Linha 4,Bebidas,13,1,158.39
4,5,Produto Carnes Linha 5,Grãos,58,1,124.43


,id,name,state,plan,created_at
0,1,Santos & Silva Atacado Distribuidora,BA,free,2023-04-17 00:16:11
1,2,Souza & Souza Comércio Distribuidora,DF,premium,2022-09-09 22:06:21
2,3,Santos & Almeida Distribuidora Distribuidora,AM,basic,2022-12-16 08:31:25
3,4,Nascimento & Ferreira Alimentos Distribuidora,GO,basic,2023-05-30 23:12:37
4,5,Silva & Santos Suprimentos Distribuidora,BA,premium,2023-03-06 10:29:33


In [8]:
# Desafio 1
query = """
WITH
  ref_data AS (
    SELECT
      MAX(created_at) AS data_max
    FROM orders
  ),

  pedidos_validos AS (
    SELECT
      o.id AS id_pedido,
      strftime('%Y-%m', o.created_at) AS mes_ano
    FROM
      orders AS o LEFT JOIN ref_data AS r
    WHERE
      o.status IN ('completed', 'delivered') AND
      o.created_at >= date(r.data_max, '-12 months')
  ),

  gmv_por_pedido AS (
    SELECT
      pv.mes_ano,
      pv.id_pedido,
      SUM(oi.qty * oi.unit_price) AS valor_bruto_pedido
    FROM
      pedidos_validos AS pv LEFT JOIN order_items AS oi ON oi.order_id = pv.id_pedido
    GROUP BY
      pv.mes_ano, pv.id_pedido
)

SELECT
  mes_ano AS mes,
  SUM(valor_bruto_pedido) AS faturamento_bruto,
  COUNT(DISTINCT id_pedido) AS qtd_pedidos,
  ROUND(SUM(valor_bruto_pedido) / NULLIF(COUNT(DISTINCT id_pedido), 0), 2) AS ticket_medio
FROM gmv_por_pedido
GROUP BY
  mes_ano
ORDER BY
  mes_ano DESC
"""

resultado_desafio1 = pysqldf(query)
resultado_desafio1


,mes,faturamento_bruto,qtd_pedidos,ticket_medio
0,2024-11,61906773.58,3643,16993.35
1,2024-10,68259702.23,3863,17670.13
2,2024-09,65922278.93,3779,17444.37
3,2024-08,65839948.65,3743,17590.15
4,2024-07,65286755.51,3862,16904.91
5,2024-06,64151214.87,3644,17604.61
6,2024-05,66388644.41,3848,17252.77
7,2024-04,63237591.49,3653,17311.14
8,2024-03,67235984.93,3877,17342.27
9,2024-02,63193936.16,3628,17418.39


In [9]:
# Desafio 2
query = """
WITH ref_trim AS (
  SELECT
    CAST(strftime('%Y', MAX(created_at)) AS INT) AS ano_ref,
    ((CAST(strftime('%m', MAX(created_at)) AS INT) + 2) / 3) AS trim_ref
  FROM orders
),
trims AS (
  SELECT
    (ano_ref * 10 + trim_ref) AS chave_trim_atual,
    CASE
      WHEN trim_ref = 1 THEN ((ano_ref - 1) * 10 + 4)
      ELSE (ano_ref * 10 + (trim_ref - 1))
    END AS chave_trim_anterior
  FROM ref_trim
),
b_pedidos AS (
  SELECT
    o.id AS id_pedido,
    o.seller_id AS id_vendedor,
    (CAST(strftime('%Y', o.created_at) AS INT) * 10) + ((CAST(strftime('%m', o.created_at) AS INT) + 2) / 3) AS chave_trim,
    SUM(oi.qty * oi.unit_price) AS gmv_pedido
  FROM
    orders AS o LEFT JOIN order_items AS oi ON oi.order_id = o.id
  WHERE
    o.status IN ('completed', 'delivered')
  GROUP BY
    o.id, o.seller_id, chave_trim
),
agg_trim_vendedor AS (
  SELECT
    id_vendedor,
    chave_trim,
    COUNT(DISTINCT id_pedido) AS qtd_pedidos,
    SUM(gmv_pedido) AS gmv_total
  FROM b_pedidos
  GROUP BY
    id_vendedor, chave_trim
),
comparativo_trims AS (
  SELECT
    v.name AS nome_vendedor,
    v.state AS estado,
    ant.gmv_total AS gmv_trim_anterior,
    atu.gmv_total AS gmv_trim_atual,
    ant.qtd_pedidos AS pedidos_trim_anterior,
    atu.qtd_pedidos AS pedidos_trim_atual
  FROM
    trims AS t LEFT JOIN agg_trim_vendedor AS atu ON atu.chave_trim = t.chave_trim_atual
               LEFT JOIN agg_trim_vendedor AS ant ON ant.id_vendedor = atu.id_vendedor AND
                                                     ant.chave_trim = t.chave_trim_anterior
               LEFT JOIN sellers AS v ON v.id = atu.id_vendedor
)
SELECT
  nome_vendedor,
  estado,
  gmv_trim_anterior,
  gmv_trim_atual,
  ROUND(((gmv_trim_atual - gmv_trim_anterior) / NULLIF(gmv_trim_anterior, 0)) * 100, 2 ) AS percentual_crescimento
FROM comparativo_trims
WHERE
  pedidos_trim_anterior >= 50 AND
  pedidos_trim_atual >= 50
ORDER BY
  percentual_crescimento DESC
LIMIT 10
"""

resultado_desafio2 = pysqldf(query)
resultado_desafio2

,nome_vendedor,estado,gmv_trim_anterior,gmv_trim_atual,percentual_crescimento
0,Rodrigues & Almeida Atacado Distribuidora,MG,947229.03,922906.69,-2.57
1,Costa & Silva Alimentos Distribuidora,BA,1094018.29,1063689.63,-2.77
2,Costa & Santos Suprimentos Distribuidora,DF,946849.42,872947.94,-7.80
3,Nascimento & Souza Alimentos Distribuidora,SP,1117256.71,1019379.83,-8.76
4,Almeida & Almeida Alimentos Distribuidora,RS,1109916.83,997166.12,-10.16
5,Almeida & Souza Mercado Distribuidora,PR,1187852.14,1055798.97,-11.12
6,Lima & Almeida Grupo Distribuidora,CE,1133549.02,1007041.67,-11.16
7,Nascimento & Almeida Alimentos Distribuidora,SP,1027109.41,909132.24,-11.49
8,Costa & Santos Atacado Distribuidora,MG,1191560.72,1043909.09,-12.39
9,Almeida & Rodrigues Distribuidora Distribuidora,RJ,1190506.71,982087.64,-17.51


In [12]:
# Desafio 3
query = """
WITH val_pedido AS (
  SELECT
    o.id AS id_pedido,
    o.created_at AS data_pedido,
    v.name AS nome_vendedor,
    SUM(oi.qty * oi.unit_price) AS valor_bruto_pedido,
    SUM(COALESCE(oi.discount, 0)) AS desconto_total_pedido
  FROM
    orders AS o LEFT JOIN order_items AS oi ON oi.order_id = o.id
                LEFT JOIN sellers AS v ON v.id = o.seller_id
  WHERE
    o.status <> 'cancelled'
  GROUP BY
    o.id, o.created_at, v.name
)
SELECT
  id_pedido,
  nome_vendedor,
  data_pedido,
  valor_bruto_pedido,
  desconto_total_pedido,
  ROUND(desconto_total_pedido / NULLIF(valor_bruto_pedido, 0), 4) AS proporcao_desconto
FROM val_pedido
WHERE
  desconto_total_pedido / NULLIF(valor_bruto_pedido, 0) > 0.40
ORDER BY
  proporcao_desconto DESC
"""

resultado_desafio3 = pysqldf(query)
resultado_desafio3

,id_pedido,nome_vendedor,data_pedido,valor_bruto_pedido,desconto_total_pedido,proporcao_desconto
0,72401,Costa & Oliveira Atacado Distribuidora,2024-01-21 02:43:43,930.30,558.02,0.5998
1,42513,Costa & Costa Atacado Distribuidora,2024-03-27 02:30:57,4082.60,2447.96,0.5996
2,36119,Costa & Costa Atacado Distribuidora,2024-11-14 01:05:01,21030.24,12598.00,0.5990
3,53223,Costa & Costa Atacado Distribuidora,2024-05-06 01:56:47,7970.40,4773.99,0.5990
4,71165,Santos & Almeida Suprimentos Distribuidora,2024-11-09 02:32:22,13385.60,8017.88,0.5990
...,...,...,...,...,...,...
896,38595,Costa & Costa Atacado Distribuidora,2023-08-21 04:52:02,27785.26,11124.91,0.4004
897,64306,Oliveira & Santos Distribuidora Distribuidora,2024-10-12 21:22:52,6907.95,2765.81,0.4004
898,32057,Souza & Ferreira Alimentos Distribuidora,2024-03-10 03:51:38,24658.56,9868.74,0.4002
899,59315,Santos & Rodrigues Distribuidora Distribuidora,2023-08-10 22:52:09,21647.49,8661.37,0.4001


In [13]:
# Desafio 4
query = """
WITH
  vendas_produto AS (
    SELECT
      product_id AS id_produto,
      SUM(qty) AS total_unidades_vendidas
    FROM order_items
    GROUP BY
      product_id
  ),

  maior_preco_por_pedido AS (
    SELECT
      order_id AS id_pedido,
      MAX(unit_price) AS maior_preco
    FROM order_items
    GROUP BY
      order_id
  ),

  produtos_com_maior_preco AS (
    SELECT DISTINCT
      oi.product_id AS id_produto
    FROM order_items AS oi
      LEFT JOIN maior_preco_por_pedido AS mpp ON mpp.id_pedido = oi.order_id AND
                                                 oi.unit_price = mpp.maior_preco
)

SELECT
  p.id AS id_produto,
  p.name AS nome_produto,
  vpp.total_unidades_vendidas
FROM
  vendas_produto AS vpp LEFT JOIN products AS p ON p.id = vpp.id_produto
                        LEFT JOIN produtos_com_maior_preco AS pmp ON pmp.id_produto = vpp.id_produto
WHERE
  vpp.total_unidades_vendidas > 1000 AND
  pmp.id_produto IS NULL
ORDER BY
  vpp.total_unidades_vendidas DESC
"""

resultado_desafio4 = pysqldf(query)
resultado_desafio4

,id_produto,nome_produto,total_unidades_vendidas
